# MR-FBP
Runtime -> Change runtime type -> GPU, then run top to bottom.
Code is pulled from GitHub; all real code lives in the `.py` files.

## Setup

In [ ]:
!pip install -q astra-toolbox scikit-image
import astra; astra.test()

In [ ]:
REPO = 'https://github.com/maradimitriu/MRFBP.git'

import os, subprocess
if not os.path.isdir('/content/mrfbp'):
    subprocess.run(['git', 'clone', REPO, '/content/mrfbp'], check=True)
%cd /content/mrfbp
!git pull --ff-only

from IPython.display import Image, display
def show(name): display(Image(filename=f'results/{name}.png'))

## Verification

In [ ]:
!python tests/test_core.py

In [ ]:
!python scripts/smoke_test.py
show('smoke_test')

In [ ]:
!python scripts/make_phantom_figure.py
show('phantom_preview')

## Experiments

In [ ]:
!python experiments/exp1_projections.py --n 256 --seeds 0 1 2 3 4
show('exp1_mae'); show('exp1_images')

In [ ]:
!python experiments/exp2_noise.py
show('exp2_noise'); show('exp2_images')

In [ ]:
!python experiments/exp3_timing.py
show('exp3_timing')

In [ ]:
!python experiments/exp4_binning.py
show('exp4_binning')

import numpy as np
d = np.load('results/exp4_binning.npz')
for k in ('n_bins', 'res', 'mae', 'ssim'):
    print(f'{k:7s}', np.round(d[k], 4))

In [ ]:
!python experiments/exp5_filters.py
show('exp5_filters')

In [ ]:
!python experiments/exp6_gradmin.py
show('exp6_gradmin'); show('exp6_lambda'); show('exp6_images')

In [ ]:
!python experiments/exp7_bases.py --seeds 0 1 2 3 4
show('exp7_bases')

In [ ]:
!python experiments/exp8_transfer.py --seeds 0 1 2
show('exp8_transfer')

import numpy as np
d = np.load('results/exp8_transfer.npz')
M, L = d['matrix'], d['labels']
print('  ' + ' '.join(f'{str(l)[:10]:>10s}' for l in L))
for i, l in enumerate(L):
    print(f'{str(l)[:14]:>14s} ' + ' '.join(f'{v:10.4f}' for v in M[i]))

In [ ]:
!python experiments/exp9_regime.py --seeds 0 1 2 3 4
show('exp9_regime'); show('exp9_slices')

## Paper-scale reproduction (N = 1024)

In [ ]:
!./run_paper_scale.sh
show('exp1_mae')

## Save the paper figures

In [ ]:
paper_figs = [
    'phantom_preview', 'exp1_mae', 'exp1_images', 'exp3_timing', 'exp5_filters',
    'exp6_gradmin', 'exp6_lambda', 'exp7_bases', 'exp8_transfer', 'exp9_regime',
]
import os, zipfile
missing = [f for f in paper_figs if not os.path.exists(f'results/{f}.png')]
assert not missing, f'not generated yet: {missing}'
with zipfile.ZipFile('/content/paper_figures.zip', 'w') as z:
    for f in paper_figs:
        z.write(f'results/{f}.png', f'{f}.png')
from google.colab import files; files.download('/content/paper_figures.zip')